# Silver flood-control component

## Purpose

Preserve every selected Bronze flood-control feature at its real source `object_id` grain. Repeated Contract IDs remain valid component evidence. This notebook safely types supported fields and retains source lineage without matching, deduplicating, or changing `silver_project`.


## 1. Confirm the Bronze safety gate

The latest terminal load and its matching Bronze validation must be safe before Silver reads the selected snapshot.


In [ ]:
%sql
USE CATALOG `buildabida-capstone`;

CREATE OR REPLACE TEMPORARY VIEW latest_flood_control_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'flood_control_projects'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'STOP: latest flood-control Bronze load is not safe for Silver'
)
FROM latest_flood_control_load;

CREATE OR REPLACE TEMPORARY VIEW latest_flood_control_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT
        *,
        DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.dq_results
    WHERE table_name = 'flood_control_projects'
      AND snapshot_id = (SELECT snapshot_id FROM latest_flood_control_load)
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: latest flood-control Bronze validation is unsafe'
)
FROM latest_flood_control_validation;


## 2. Preserve one row per source feature

Contract normalization uses only `TRIM` and `UPPER`. Punctuation is preserved. Invalid numeric values stay visible through raw fields and parse-status columns.


In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW flood_control_component_input AS
SELECT
    CAST(object_id AS STRING) AS source_feature_id,
    contract_id AS source_contract_id_raw,
    CASE
        WHEN contract_id IS NULL OR TRIM(contract_id) = '' THEN NULL
        ELSE UPPER(TRIM(contract_id))
    END AS source_contract_id_normalized,
    project_id AS source_project_id,
    project_component_id AS source_project_component_id,
    description AS component_description_raw,
    infra_type AS source_infra_type_raw,
    type_of_work AS source_type_of_work_raw,
    reported_region AS reported_region_raw,
    reported_province AS reported_province_raw,
    reported_municipality AS reported_municipality_raw,
    deo AS district_engineering_office_raw,
    abc AS approved_budget_cost_raw,
    contract_cost AS source_contract_cost_raw,
    TRY_CAST(TRIM(contract_cost) AS DECIMAL(20, 2)) AS source_contract_cost_pesos,
    CASE
        WHEN contract_cost IS NULL OR TRIM(contract_cost) = '' THEN 'MISSING'
        WHEN TRY_CAST(TRIM(contract_cost) AS DECIMAL(20, 2)) IS NULL THEN 'UNPARSEABLE'
        WHEN TRY_CAST(TRIM(contract_cost) AS DECIMAL(20, 2)) < 0 THEN 'NEGATIVE'
        ELSE 'VALID'
    END AS source_contract_cost_status,
    TRY_CAST(infra_year AS INT) AS infrastructure_year,
    start_date AS start_date_raw,
    TRY_CAST(start_date AS DATE) AS start_date_parsed,
    completion_date AS completion_date_raw,
    TRY_CAST(completion_date AS DATE) AS completion_date_parsed,
    contractor AS contractor_raw,
    latitude AS latitude_raw,
    TRY_CAST(latitude AS DOUBLE) AS latitude_parsed,
    longitude AS longitude_raw,
    TRY_CAST(longitude AS DOUBLE) AS longitude_parsed,
    CASE
        WHEN (latitude IS NULL OR TRIM(latitude) = '')
         AND (longitude IS NULL OR TRIM(longitude) = '') THEN 'MISSING_PAIR'
        WHEN latitude IS NULL OR TRIM(latitude) = ''
          OR longitude IS NULL OR TRIM(longitude) = '' THEN 'PARTIAL_PAIR'
        WHEN TRY_CAST(latitude AS DOUBLE) IS NULL
          OR TRY_CAST(longitude AS DOUBLE) IS NULL THEN 'UNPARSEABLE'
        ELSE 'VALID_PAIR'
    END AS coordinate_status,
    source_record_json,
    geometry_json,
    _source_system AS source_system,
    _source_path AS source_path,
    _source_file AS source_file,
    _source_snapshot_id AS source_snapshot_id,
    _ingest_run_id AS source_ingest_run_id,
    _ingested_at AS source_load_ts,
    latest.source_version,
    SHA2(
        CONCAT_WS(
            '||',
            COALESCE(_source_snapshot_id, ''),
            COALESCE(_ingest_run_id, ''),
            COALESCE(latest.source_version, ''),
            'silver_flood_control_component_v1'
        ),
        256
    ) AS run_id,
    'silver_flood_control_component_v1' AS transformation_rule_version
FROM `01-bronze`.flood_control_projects
CROSS JOIN latest_flood_control_load AS latest
WHERE _source_snapshot_id = latest.snapshot_id;

CREATE OR REPLACE TABLE `02-silver`.silver_flood_control_component AS
SELECT
    SHA2(
        CONCAT_WS(
            '||',
            COALESCE(source_system, ''),
            COALESCE(source_snapshot_id, ''),
            COALESCE(source_feature_id, ''),
            transformation_rule_version
        ),
        256
    ) AS flood_component_key,
    *
FROM flood_control_component_input;

SELECT ASSERT_TRUE(
    (SELECT COUNT(*) FROM `02-silver`.silver_flood_control_component)
    = (
        SELECT COUNT(*)
        FROM `01-bronze`.flood_control_projects
        WHERE _source_snapshot_id = (SELECT snapshot_id FROM latest_flood_control_load)
    ),
    'STOP: flood-control Bronze-to-Silver row preservation failed'
);


# Summary

Created `02-silver`.silver_flood_control_component at one selected Bronze source feature per row. The table uses the real `object_id`, preserves repeated Contract IDs and raw cost evidence, and performs no project matching or budget aggregation.

Requires Databricks execution.
